# 09 — Robustness for the CHASE Revision  
### *Coding Alone* pipeline

**Self-contained.** Set PROJECT_ID, then **Runtime → Run all**. Cheap (reads your own tables only).

Produces every number the revised manuscript needs, in order:
1. DiD with **95% CIs** and the largest differential effect the CI cannot exclude.
2. **Exhaustive permutation (randomization) inference** over which 3 of 14 languages are treated (all 364 assignments), robust to the few-clusters problem.
3. **Treatment-definition variants**: top-3 (base), top-5 (adds Java, C#), and boundary-exclusion (drops Java, C#, Go from controls).
4. **Alternative AI break dates** for the ITS: Copilot preview (2021-06), Copilot GA (2022-06), ChatGPT (2022-11).
5. **NRR 12-month washout**: newcomers first active ≥ Jan 2019 only (kills the left-censoring threat).
6. **End-date sensitivity**: ITS re-fit with the window ending 2024-12.
7. Share of frame repos with **≥2 distinct contributors** (for the population-definition fix).
8. **Within-slope sanity re-print** (units/coding verification).
9. **Corrected Figure 1** (NRR panel truncated at 2024-12).

All summaries print AND save to `Drive/ICSE_SEIS/tables/robustness_*.csv`. Paste the printed blocks back to me.


## Config + setup

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
BQ_DATASET="seis"; BQ_LOCATION="US"; DRIVE_BASE="/content/drive/MyDrive/ICSE_SEIS"
WIN_START="201801"; WIN_END="202506"; MDI_END="202412"; GAP=["202110"]
HIGH3=["Python","JavaScript","TypeScript"]; HIGH5=HIGH3+["Java","C#"]; BOUNDARY=["Java","C#","Go"]
AI_DATES={"copilot_preview":"202106","copilot_ga":"202206","chatgpt":"202211"}
import numpy as np, pandas as pd, itertools, os, json
import statsmodels.formula.api as smf, statsmodels.api as sm
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from google.colab import auth, drive
from google.cloud import bigquery
auth.authenticate_user(); client=bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive'); os.makedirs(DRIVE_BASE+"/tables", exist_ok=True)
P=f"{PROJECT_ID}.{BQ_DATASET}"
def q(s): return client.query(s).result().to_dataframe()
def ymidx(s): return (s.str[:4].astype(int)-2018)*12 + (s.str[4:].astype(int)-1)
METRICS=["cd","swr","cr","hie","nrr"]
print("Ready.", P)


## Data pulls (same definitions as the paper)

In [ ]:
lm = q(f"""
SELECT l.language, p.ym,
  AVG(IF(p.active,p.cd,NULL)) cd, AVG(p.swr) swr, AVG(p.reciprocity) cr, AVG(p.hie_norm) hie, COUNT(*) n
FROM `{P}.metrics_panel` p JOIN `{P}.sampling_frame_enriched` l USING(repo_id)
WHERE p.ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND p.ym NOT IN UNNEST({GAP})
GROUP BY language, ym""")
lm_nrr = q(f"""
SELECT l.language, m.ym, AVG(IF(m.responses>0,1.0,0)) nrr
FROM `{P}.mdi_newcomers` m JOIN `{P}.sampling_frame_enriched` l USING(repo_id)
WHERE m.ym BETWEEN '{WIN_START}' AND '{MDI_END}' AND m.ym NOT IN UNNEST({GAP})
GROUP BY language, ym""")
lm = lm.merge(lm_nrr, on=["language","ym"], how="left")
keep=[x for x in lm.language.value_counts().index if x!="unknown"][:14]
lm = lm[lm.language.isin(keep)].copy(); lm["t"]=ymidx(lm.ym)
LANGS=sorted(lm.language.unique()); print(len(LANGS),"languages:",LANGS)

eco = q(f"""
SELECT ym, AVG(IF(active,cd,NULL)) cd, AVG(swr) swr, AVG(reciprocity) cr, AVG(hie_norm) hie
FROM `{P}.metrics_panel` WHERE ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND ym NOT IN UNNEST({GAP})
GROUP BY ym ORDER BY ym""")
nrr_eco = q(f"""SELECT ym, AVG(IF(responses>0,1.0,0)) nrr FROM `{P}.mdi_newcomers`
WHERE ym BETWEEN '{WIN_START}' AND '{MDI_END}' AND ym NOT IN UNNEST({GAP}) GROUP BY ym ORDER BY ym""")
eco = eco.merge(nrr_eco,on="ym",how="left"); eco["t"]=ymidx(eco.ym); eco["mo"]=eco.ym.str[4:]
print("panel:",lm.shape,"| eco:",eco.shape)


## 1+2+3. DiD: CIs, exhaustive permutation inference, treatment variants

In [ ]:
def did_fit(d, metric, treated, post="202211", drop=None):
    dd = d.dropna(subset=[metric]).copy()
    if drop: dd = dd[~dd.language.isin(drop)]
    dd["treatpost"] = (dd.language.isin(treated) & (dd.ym>=post)).astype(float)
    m = smf.ols(f"{metric} ~ C(language)+C(ym)+treatpost", data=dd).fit(
        cov_type="cluster", cov_kwds={"groups":dd["language"]})
    lo,hi = m.conf_int().loc["treatpost"]
    return m.params["treatpost"], m.bse["treatpost"], m.pvalues["treatpost"], lo, hi, dd

rows=[]
for met in METRICS:
    coef,se,p,lo,hi,dd = did_fit(lm, met, HIGH3)
    # exhaustive permutation over which 3 languages are treated (364 assignments)
    perm=[]
    for combo in itertools.combinations(LANGS,3):
        dd2 = dd.copy(); dd2["treatpost"]=(dd2.language.isin(combo)&(dd2.ym>="202211")).astype(float)
        mm = smf.ols(f"{met} ~ C(language)+C(ym)+treatpost", data=dd2).fit()
        perm.append(mm.params["treatpost"])
    perm=np.array(perm); perm_p = float(np.mean(np.abs(perm)>=abs(coef)))
    c5,_,p5,lo5,hi5,_ = did_fit(lm, met, HIGH5)
    cb,_,pb,lob,hib,_ = did_fit(lm, met, HIGH3, drop=BOUNDARY)
    rows.append(dict(metric=met, did=round(coef,5), ci_lo=round(lo,5), ci_hi=round(hi,5),
                     cluster_p=round(p,3), perm_p=round(perm_p,3),
                     excl_bound=round(max(abs(lo),abs(hi)),5),
                     top5_did=round(c5,5), top5_p=round(p5,3),
                     noboundary_did=round(cb,5), noboundary_p=round(pb,3)))
did_tab=pd.DataFrame(rows); did_tab.to_csv(f"{DRIVE_BASE}/tables/robustness_did.csv",index=False)
print("=== DiD: CI + permutation + variants ==="); print(did_tab.to_string(index=False))


## 4. ITS with alternative AI break dates (COVID break kept)

In [ ]:
def its(d, metric, ai_ym):
    dd=d.dropna(subset=[metric]).copy()
    for br,name in [("202003","covid"),(ai_ym,"ai")]:
        b=(int(br[:4])-2018)*12+(int(br[4:])-1)
        dd[f"{name}_lvl"]=(dd.t>=b).astype(int); dd[f"{name}_slope"]=np.maximum(0,dd.t-b)
    m=smf.ols(f"{metric} ~ t+covid_lvl+covid_slope+ai_lvl+ai_slope+C(mo)",data=dd
             ).fit(cov_type="HAC",cov_kwds={"maxlags":12})
    return m.params["ai_lvl"],m.pvalues["ai_lvl"],m.params["ai_slope"],m.pvalues["ai_slope"]
rows=[]
for met in METRICS:
    for name,ymd in AI_DATES.items():
        al,pl,asl,ps = its(eco, met, ymd)
        rows.append(dict(metric=met, break_date=name, ai_level=round(al,5), p_lvl=round(pl,3),
                         ai_slope=round(asl,5), p_slope=round(ps,3)))
its_tab=pd.DataFrame(rows); its_tab.to_csv(f"{DRIVE_BASE}/tables/robustness_breakdates.csv",index=False)
print("=== ITS alternative break dates ==="); print(its_tab.to_string(index=False))


## 5. NRR 12-month washout (newcomers first active ≥ Jan 2019)

In [ ]:
wash = q(f"""
WITH fe AS (SELECT * FROM `{P}.frame_events_*`),
acts AS (
  SELECT repo_id, actor_login, MIN(created_at) first_seen FROM fe
  WHERE NOT is_bot AND actor_login IS NOT NULL
    AND type IN ('PushEvent','PullRequestEvent','IssuesEvent','IssueCommentEvent',
                 'PullRequestReviewCommentEvent','CommitCommentEvent')
  GROUP BY 1,2),
opens AS (
  SELECT repo_id, actor_login newcomer, number, MIN(created_at) open_time FROM fe
  WHERE type IN ('PullRequestEvent','IssuesEvent') AND action='opened' AND NOT is_bot AND number IS NOT NULL
  GROUP BY 1,2,3),
first_thread AS (
  SELECT repo_id,newcomer,number,open_time FROM (
    SELECT o.*, ROW_NUMBER() OVER (PARTITION BY repo_id,newcomer ORDER BY open_time) rn FROM opens o) WHERE rn=1),
newc AS (
  SELECT ft.* FROM first_thread ft JOIN acts a ON ft.repo_id=a.repo_id AND ft.newcomer=a.actor_login
  WHERE ft.open_time <= TIMESTAMP_ADD(a.first_seen, INTERVAL 30 DAY)
    AND a.first_seen >= TIMESTAMP('2019-01-01'))          -- 12-month washout
SELECT FORMAT_TIMESTAMP('%Y', n.open_time) yr, COUNT(*) newcomers,
       AVG(IF(cnt>0,1.0,0)) nrr
FROM (
  SELECT n.repo_id,n.newcomer,n.number,n.open_time,
         COUNTIF(e.actor_login!=n.newcomer AND NOT e.is_bot) cnt
  FROM newc n LEFT JOIN fe e ON e.repo_id=n.repo_id AND e.number=n.number
    AND e.type IN ('IssueCommentEvent','PullRequestReviewCommentEvent')
    AND e.created_at BETWEEN n.open_time AND TIMESTAMP_ADD(n.open_time, INTERVAL 90 DAY)
  GROUP BY 1,2,3,4) n
WHERE FORMAT_TIMESTAMP('%Y%m', n.open_time) BETWEEN '201901' AND '{MDI_END}'
GROUP BY yr ORDER BY yr""")
wash.to_csv(f"{DRIVE_BASE}/tables/robustness_nrr_washout.csv",index=False)
print("=== NRR yearly, 12-month washout cohort ==="); print(wash.to_string(index=False))


## 6. End-date sensitivity (window ending 2024-12)

In [ ]:
eco24 = eco[eco.ym<="202412"].copy()
rows=[]
for met in METRICS:
    al,pl,asl,ps = its(eco24, met, "202211")
    d=eco24.dropna(subset=[met]); sl=smf.ols(f"{met} ~ t",data=d).fit(cov_type="HAC",cov_kwds={"maxlags":12})
    rows.append(dict(metric=met, lin_slope=round(sl.params["t"],5), p=round(sl.pvalues["t"],4),
                     ai_level=round(al,5), p_lvl=round(pl,3), ai_slope=round(asl,5), p_slope=round(ps,3)))
end_tab=pd.DataFrame(rows); end_tab.to_csv(f"{DRIVE_BASE}/tables/robustness_end2024.csv",index=False)
print("=== Window ending 2024-12 ==="); print(end_tab.to_string(index=False))


## 7+8. Frame ≥2 contributors share + within-slope sanity re-print

In [ ]:
multi = q(f"SELECT COUNTIF(peak_year_actors>=2)/COUNT(*) share2, COUNT(*) n FROM `{P}.sampling_frame_enriched`")
print(f"Share of frame repos with >=2 distinct contributors (peak-year proxy): {float(multi.share2.iloc[0]):.3f}")

panel = q(f"""
WITH act AS (SELECT repo_id, COUNTIF(active) m FROM `{P}.metrics_panel`
  WHERE ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND ym NOT IN UNNEST({GAP}) GROUP BY repo_id),
samp AS (SELECT repo_id FROM act WHERE m>=24 ORDER BY FARM_FINGERPRINT(CAST(repo_id AS STRING)) LIMIT 8000)
SELECT p.repo_id,p.ym,IF(p.active,p.cd,NULL) cd,p.swr,p.reciprocity cr,p.hie_norm hie
FROM `{P}.metrics_panel` p JOIN samp USING(repo_id)
WHERE p.ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND p.ym NOT IN UNNEST({GAP})""")
panel["t"]=ymidx(panel.ym)
print("t coding check: min",panel.t.min(),"max",panel.t.max(),"(months since 2018-01)")
for met in ["cd","swr","cr","hie"]:
    d=panel.dropna(subset=[met])[["repo_id","t",met]].copy()
    d["y"]=d[met]-d.groupby("repo_id")[met].transform("mean")
    d["x"]=d["t"]-d.groupby("repo_id")["t"].transform("mean")
    m=sm.OLS(d["y"],sm.add_constant(d["x"])).fit(cov_type="cluster",cov_kwds={"groups":d["repo_id"]})
    print(f"within {met}: slope/month={m.params['x']:.6f}  p={m.pvalues['x']:.5f}  repos={d.repo_id.nunique()}  obs={len(d)}")


## 9. Corrected Figure 1 (NRR truncated at 2024-12)

In [ ]:
eco["date"]=pd.to_datetime(eco.ym+"01",format="%Y%m%d")
plt.rcParams.update({"font.size":12,"axes.titlesize":11.5,"xtick.labelsize":10,"ytick.labelsize":10,
                     "axes.spines.top":False,"axes.spines.right":False})
OK={"swr":"#0072B2","hie":"#009E73","cr":"#E69F00","nrr":"#CC79A7","cd":"#D55E00"}
specs=[("swr","Solitary Workflow Ratio"),("hie","Human Interaction Entropy"),
       ("cr","Communication Reciprocity"),("nrr","Newcomer Response Rate"),("cd","Collaboration Density")]
covid,ai=pd.Timestamp("2020-03-01"),pd.Timestamp("2022-11-01")
fig,axes=plt.subplots(1,5,figsize=(15,2.9))
for ax,(col,title) in zip(axes,specs):
    d=eco.dropna(subset=[col])
    if col=="nrr": d=d[d.ym<="202412"]          # trust window for NRR
    ax.axvspan(ai,d["date"].max(),color="0.92",zorder=0)
    ax.plot(d["date"],d[col],color=OK[col],lw=1.8)
    ax.axvline(covid,ls=":",color="0.55",lw=1); ax.axvline(ai,ls="--",color="0.25",lw=1)
    ax.set_title(title,pad=6); ax.grid(axis="y",alpha=.25); ax.margins(x=0.02)
    ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x",length=0)
fig.tight_layout(w_pad=1.3)
for ext in ("pdf","png"): fig.savefig(f"{DRIVE_BASE}/figures/fig1_trends.{ext}",dpi=300,bbox_inches="tight")
plt.show(); print("Saved corrected fig1_trends (NRR ends 2024-12).")


## Done ✅ — paste back every printed block above.

In [ ]:
# Deterministic language set + final DiD block (supersedes earlier DiD output)
top_lang = q(f"""SELECT language, COUNT(*) n FROM `{P}.sampling_frame_enriched`
WHERE language != 'unknown' GROUP BY language ORDER BY n DESC, language LIMIT 14""")
LANGS = sorted(top_lang.language.tolist()); print("Pinned languages:", LANGS)
lm2 = lm[lm.language.isin(LANGS)].copy()
rows=[]
for met in METRICS:
    coef,se,p,lo,hi,dd = did_fit(lm2, met, HIGH3)
    perm=[]
    for combo in itertools.combinations(LANGS,3):
        dd2=dd.copy(); dd2["treatpost"]=(dd2.language.isin(combo)&(dd2.ym>="202211")).astype(float)
        perm.append(smf.ols(f"{met} ~ C(language)+C(ym)+treatpost",data=dd2).fit().params["treatpost"])
    perm=np.array(perm)
    c5,_,p5,_,_,_ = did_fit(lm2, met, HIGH5)
    cb,_,pb,_,_,_ = did_fit(lm2, met, HIGH3, drop=BOUNDARY)
    rows.append(dict(metric=met, did=round(coef,5), ci_lo=round(lo,5), ci_hi=round(hi,5),
                     cluster_p=round(p,3), perm_p=round(float(np.mean(np.abs(perm)>=abs(coef))),3),
                     top5_did=round(c5,5), top5_p=round(p5,3),
                     noboundary_did=round(cb,5), noboundary_p=round(pb,3)))
final=pd.DataFrame(rows); final.to_csv(f"{DRIVE_BASE}/tables/robustness_did_FINAL.csv",index=False)
print(final.to_string(index=False))

In [ ]:
import warnings; warnings.filterwarnings("ignore")
# A-priori universe: general-purpose programming languages only
# (markup: HTML, CSS; framework: Vue; notebook format: Jupyter Notebook -- excluded on construct grounds)
PROG = ['C','C#','C++','Go','Java','JavaScript','PHP','Python','Ruby','TypeScript']
lm3 = lm[lm.language.isin(PROG)].copy(); LANGS3 = sorted(PROG)
rows=[]
for met in METRICS:
    coef,se,p,lo,hi,dd = did_fit(lm3, met, HIGH3)
    perm=[]
    for combo in itertools.combinations(LANGS3,3):
        dd2=dd.copy(); dd2["treatpost"]=(dd2.language.isin(combo)&(dd2.ym>="202211")).astype(float)
        perm.append(smf.ols(f"{met} ~ C(language)+C(ym)+treatpost",data=dd2).fit().params["treatpost"])
    perm=np.array(perm)
    rows.append(dict(metric=met, did=round(coef,5), ci_lo=round(lo,5), ci_hi=round(hi,5),
                     cluster_p=round(p,3), perm_p=round(float(np.mean(np.abs(perm)>=abs(coef))),3)))
prog_tab=pd.DataFrame(rows); prog_tab.to_csv(f"{DRIVE_BASE}/tables/robustness_did_PROGLANGS.csv",index=False)
print("Universe: 10 general-purpose languages (3 treated, 7 control)")
print(prog_tab.to_string(index=False))

## DiD validity checks on the a-priori 10-language universe
Placebo treatment (2021-06, pre-period only) and differential pre-trend test.
These produce the values reported in the paper's Table 3 caption and RQ3 text.

In [ ]:
import statsmodels.formula.api as smf
PROG=['C','C#','C++','Go','Java','JavaScript','PHP','Python','Ruby','TypeScript']
HIGH=['Python','JavaScript','TypeScript']
# `lm` is the language-month panel built in the data-pull cell above;
# offline, load data/language_month_panel.csv instead.
pre=lm[lm.language.isin(PROG)&(lm.ym<'202211')].copy()
pre['high']=pre.language.isin(HIGH).astype(float)
yms=sorted(pre.ym.unique()); tmap={y:i for i,y in enumerate(yms)}
pre['t']=pre.ym.map(tmap)
print(f"{'metric':6} {'placebo_coef':>13} {'placebo_p':>10} {'pretrend_coef':>14} {'pretrend_p':>11}")
for metric in ['cd','swr','cr','hie','nrr']:
    d1=pre.dropna(subset=[metric]).copy()
    d1['treatpost']=(d1.high*(d1.ym>='202106')).astype(float)
    m1=smf.ols(f"{metric} ~ C(language)+C(ym)+treatpost",data=d1).fit(
        cov_type='cluster',cov_kwds={'groups':d1['language']})
    m2=smf.ols(f"{metric} ~ C(language)+t+t:high",data=d1).fit(
        cov_type='cluster',cov_kwds={'groups':d1['language']})
    print(f"{metric:6} {m1.params['treatpost']:+13.5f} {m1.pvalues['treatpost']:10.3f} "
          f"{m2.params['t:high']:+14.5f} {m2.pvalues['t:high']:11.3f}")
